# 02 — Getting Knowledge Out of Our Code

Notebook 01 ended on a problem: our stories were typed straight into a Python dict, tangled up with the code that calls the model. Fix a typo in Baahubali's plot, and you're editing the same file that talks to Groq.

Today we untangle that. Knowledge moves into files; code just reads them. A librarian doesn't memorize every book in the building — they know where the books are. That's the move we're making.


In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"

DATA_DIR = os.path.join("..", "data")
STORIES_DIR = os.path.join(DATA_DIR, "stories")


## Setting up the data folder

We'll write our 5 StoryVerse stories to disk, once, as plain `.txt` files. From here on, these files — not Python variables — are the source of truth.


In [ ]:
os.makedirs(STORIES_DIR, exist_ok=True)

stories = {
    "interstellar.txt": """
Cooper, a former NASA pilot turned farmer, discovers a hidden NASA
facility and is recruited for a mission to save humanity from a dying
Earth. Traveling through a wormhole near Saturn, his crew searches
distant planets for a new home. Time dilation near a black hole costs
Cooper decades with his daughter Murph, who grows up to help solve the
gravitational equation that could save Earth. Cooper eventually falls
into the black hole itself, ending up inside a tesseract built by
future humans -- a structure that lets him reach across time and send
Murph the data she needs, saving humanity from the dust and famine
that were slowly killing it.
""",
    "harry_potter_sorcerers_stone.txt": """
Harry Potter, an orphan raised by unkind relatives, learns on his
eleventh birthday that he is a wizard and is enrolled at Hogwarts
School of Witchcraft and Wizardry. He makes friends with Ron Weasley
and Hermione Granger, learns his parents were killed by the dark
wizard Voldemort, and discovers that someone is trying to steal the
Sorcerer's Stone, an object capable of granting immortality, hidden
somewhere in the school. Harry and his friends work out that
Voldemort, weakened but alive, is behind it, and stop him from
obtaining the Stone in a final confrontation beneath the school.
""",
    "baahubali.txt": """
Shivudu grows up in a remote village, unaware that he is the son of a
murdered king. After rescuing a woman thrown over a waterfall, he
climbs the cliff she came from out of curiosity and falls in love with
Avanthika, a warrior on a secret mission. He follows her to the
kingdom of Mahishmati, where he learns of his true heritage, his
father's death, and the betrayal of his uncle Bhallaladeva. Taking up
his father's mace, he rallies the kingdom's oppressed people and moves
to reclaim the throne that was always his by birth.
""",
    "portal_bookshop.txt": """
Arjun had walked past the same bookshop in Hyderabad's Abids market a
hundred times without a second glance -- dusty windows, a hand-painted
sign, nothing special. One rainy evening the door was open just a
crack, spilling warm light onto the street. Inside, past shelves of
yellowed paperbacks, stood a second doorway that hadn't been there
before, its outline shimmering like heat over summer asphalt. The
moment Arjun crossed it, the smell of old paper vanished, replaced by
salt air and the sound of waves -- he was standing on a beach under
two moons, the bookshop gone behind him, with a set of footprints in
the sand that weren't his own.
""",
    "dark_knight.txt": """
Batman, District Attorney Harvey Dent, and Commissioner Gordon attempt
to dismantle organized crime in Gotham City. The Joker, an anarchic
criminal mastermind with no interest in money or power, escalates the
chaos, engineering a series of impossible choices that push Gotham to
its breaking point. After a personal tragedy, Harvey Dent is
transformed into the vengeful Two-Face. To protect Gotham's faith in
its "white knight," Batman takes the blame for Dent's crimes himself,
becoming a fugitive hunted by the very police force he was trying to
help.
""",
}

for filename, content in stories.items():
    with open(os.path.join(STORIES_DIR, filename), "w") as f:
        f.write(content.strip())

print("Files written:", sorted(os.listdir(STORIES_DIR)))


## Loading them back — pure Python

No frameworks yet. Just read every `.txt` file in the folder.


In [ ]:
def load_documents(folder_path: str) -> list[dict]:
    """Read every .txt file in a folder into {"filename", "content"} dicts."""
    docs = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename)) as f:
                docs.append({"filename": filename, "content": f.read()})
    return docs


def combine_documents(docs: list[dict]) -> str:
    """Join every document's content into one block, separated and labeled."""
    return "\n\n".join(f"[{d['filename']}]\n{d['content']}" for d in docs)


docs = load_documents(STORIES_DIR)
combined = combine_documents(docs)
print(f"Loaded {len(docs)} documents, {len(combined):,} combined characters")
print("\nPreview:\n", combined[:200], "...")


Clean. But notice what just happened: we loaded and combined **every single story**, even though the question we're about to ask only needs one of them. That's fine for 5 files. Keep that thought — notebook 03 is going to make you feel exactly why it isn't fine for 500.


## The full pipeline, still pure Python


In [ ]:
def answer_question(question: str, docs_folder: str) -> str:
    docs = load_documents(docs_folder)
    context = combine_documents(docs)

    system_prompt = f"""You are StoryVerse AI. Answer using ONLY the context below.

Context:
{context}"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


test_questions = [
    "What happens to Arjun in the bookshop story?",
    "Who is Cooper's daughter in Interstellar?",
    "Why does Batman take the blame for Harvey Dent's crimes?",
]

for q in test_questions:
    print("Q:", q)
    print("A:", answer_question(q, STORIES_DIR))
    print()


## Enter LangChain

Every one of the pieces above — reading files, building a prompt, calling a model — is something LangChain also does, wrapped in reusable classes. It's worth being blunt about what LangChain actually is: an **orchestration framework**. It does not make the model smarter. It gives you pre-built pieces (loaders, prompt templates, chains) instead of writing `load_documents` yourself every time.

Think of it like Express.js for Node — you could write a raw HTTP server by hand, and Express doesn't change what HTTP is, it just gives you structure around it. Same deal here: LangChain doesn't change what an LLM call is, it gives you structure around it.

The one warning worth remembering: a lot of people treat LangChain as magic. It isn't. We'll show the plain-Python equivalent of everything below, side by side, so nothing stays a black box.


### Document loading, the LangChain way


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(STORIES_DIR, glob="*.txt", loader_cls=TextLoader)
lc_docs = loader.load()

print(f"LangChain loaded {len(lc_docs)} documents")
print("\nFirst doc metadata:", lc_docs[0].metadata)
print("First doc content preview:", lc_docs[0].page_content[:120], "...")


Compare that to our own `load_documents()` output above — same information, wrapped in a `Document` object instead of a plain dict. `page_content` is our `content`, `metadata` is where the filename lives. No magic, just a different shape.


### Prompt templates, the LangChain way


In [ ]:
from langchain_core.prompts import PromptTemplate

# Our version, from notebook 01:
our_prompt = f"Context:\n{combined[:60]}...\n\nQuestion:\nWho is Arjun?"

# LangChain's version of the exact same idea:
template = PromptTemplate(
    input_variables=["context", "question"],
    template="Context:\n{context}\n\nQuestion:\n{question}",
)
lc_prompt = template.format(context=combined[:60] + "...", question="Who is Arjun?")

print("Ours:\n", our_prompt)
print("\nLangChain's:\n", lc_prompt)
print("\nIdentical output:", our_prompt == lc_prompt)


`PromptTemplate` is a string formatter with input validation. That's genuinely the whole thing.


### Chains, the LangChain way

A **chain** is just: take an input, transform it, pass it to the next step, return the output. Here we wire a prompt template straight into the model using LangChain's pipe syntax (`|`) — this style is called **LCEL** (LangChain Expression Language).


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model=MODEL, temperature=0.2)

rag_template = PromptTemplate(
    input_variables=["context", "question"],
    template="You are StoryVerse AI. Answer using ONLY this context.\n\nContext:\n{context}\n\nQuestion:\n{question}",
)

chain = rag_template | llm

result = chain.invoke({"context": combined, "question": "Who is Cooper's daughter in Interstellar?"})
print("LangChain answer:", result.content)
print("\nOur pure-Python answer:", answer_question("Who is Cooper's daughter in Interstellar?", STORIES_DIR))


Same answer, either way. `prompt | llm` means "format the prompt, then feed it to the model" — you could write that exact sequence yourself in about 10 lines, which you basically already did in notebook 01.


## So why bother with LangChain at all?

If it's not smarter and not magic, what's the actual point?

- **Reusability** — swap `ChatGroq` for a different provider without rewriting your prompt logic
- **Composability** — chains plug into bigger chains
- **Ecosystem** — loaders and integrations for almost any data source already exist, so you're not writing a PDF parser from scratch
- **Observability** — tools like LangSmith (notebook 10) plug straight into this

Use it as scaffolding once you understand what it's scaffolding. That's the whole reason we built the raw version first.


## The problem we just re-introduced

We now load **every** document, for **every** question, no matter how small the question is. Let's watch that cost curve again, the same way we did in notebook 01 — except now it's file I/O and prompt-building cost, not just prompt size.


In [ ]:
avg_story_chars = len(combined) // len(docs)

print(f"{'Docs':>6} | {'Combined chars':>15} | {'~Tokens':>10} | {'Est. cost/query*':>18}")
for num_docs in [5, 25, 100, 1000]:
    total_chars = num_docs * avg_story_chars
    est_tokens = total_chars // 4
    est_cost = est_tokens / 1_000_000 * 0.10  # rough $0.10 / 1M tokens, illustrative only
    print(f"{num_docs:>6} | {total_chars:>15,} | {est_tokens:>10,} | ${est_cost:>17.4f}")

print("\n*Illustrative pricing, not a real quote -- the point is the shape of the curve.")


At 1,000 stories, we're paying real money and burning real latency to answer a question that only ever needed one file — and we're sending Batman trivia to a question about Hyderabad. We don't need to load *everything*. We need to load *the right thing*. Knowing which document is "right" for a given question is the whole problem embeddings and retrieval exist to solve — but before we get there, notebook 03 is going to make you actually measure how bad "load everything" gets, so retrieval feels necessary instead of optional.


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Document loader | Code that reads raw files into a standard `{content, metadata}` shape |
| Prompt template | A string formatter with named slots (`{context}`, `{question}`) |
| Chain | Input → transform → next step → output, wired together |
| LCEL | LangChain's `prompt \| llm` pipe syntax for building chains |
| Orchestration | Coordinating existing pieces (loaders, prompts, models) — not intelligence itself |

Knowledge now lives in files, not in code. We've seen LangChain do the exact same steps we did by hand, with a nicer API. And we've re-confirmed that "load everything, every time" doesn't scale.

**Exercises:**

- Add a 6th story file to `STORIES_DIR`, re-run `load_documents`, and confirm it's picked up automatically.
- Print `lc_docs[0].__dict__` and compare its shape to our own `{"filename": ..., "content": ...}` dict.
- Break the LangChain version on purpose (misspell `input_variables`) and read the error — it tells you exactly what `PromptTemplate` is checking for internally.
